# EEC v5 — Analysis
**Emergent Expansion Cosmology (Ismail Khan 2026)**

https://doi.org/10.5281/zenodo.19830980

ismailkhan.mcs@gmail.com

ismailkhan.webdev@gmail.com

Copyright 2026 Ismail Khan. All rights reserved.

Omega_c derived from peak halo formation rate (z_halo = 1.1, Fakhouri+ 2010).
Three free parameters: omega_m, H0, sigma_8 (same count as LCDM).

Datasets: Pantheon+ (1701 SNe, full cov) + BOSS DR12 (BAO+fsig8) + DESI DR1 (7 BAO bins) + Planck omega_m.

theta* computed using r_s(z_star) and full EEC E(z).

In [ ]:
# Cell 1 — Setup
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'emcee', 'corner', '-q'])
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CKPT_DIR = '/content/drive/MyDrive/EEC_V5'
    os.makedirs(CKPT_DIR, exist_ok=True)
except Exception:
    CKPT_DIR = 'ckpt_EEC_V5'
    os.makedirs(CKPT_DIR, exist_ok=True)
import emcee, numpy as np
print(f'emcee {emcee.__version__}, checkpoints: {CKPT_DIR}')


In [ ]:
# Cell 2 — Constants
c_km = 299792.458
T_CMB = 2.7255
omega_gam = 2.469e-5
N_eff = 3.046
omega_nu = (7./8.)*(4./11.)**(4./3.)*N_eff*omega_gam
omega_r = omega_gam + omega_nu
omega_b = 0.02166
omega_b_err = 0.00015
delta_c = 1.686
omega_m_cmb = 0.1430
omega_m_cmb_err = 0.0011
theta_star = 0.010409
theta_star_err = 0.000031

# Peak halo formation redshift (Fakhouri+ 2010, MNRAS 406, 2267)
Z_HALO = 1.1
ZHALO_FACTOR = (1 + Z_HALO)**3  # = 9.261

print(f'z_halo = {Z_HALO}')
print(f'Omega_c = Omega_m * (1+z_halo)^3 = Omega_m * {ZHALO_FACTOR:.3f}')


In [ ]:
# Cell 3 — Pantheon+
import urllib.request
BASE = ('https://raw.githubusercontent.com/PantheonPlusSH0ES/'
        'DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/')
DAT = 'Pantheon+SH0ES.dat'
COV = 'Pantheon+SH0ES_STAT+SYS.cov'
for f, u in [(DAT, BASE+DAT), (COV, BASE+COV)]:
    if not os.path.exists(f):
        print(f'Downloading {f}...', end=' ')
        urllib.request.urlretrieve(u, f)
        print('done')
pp = np.genfromtxt(DAT, names=True, dtype=None, encoding=None)
mask = (pp['IS_CALIBRATOR']==0) & (pp['zHD'].astype(float)>0.01)
z_sn = pp['zHD'][mask].astype(float)
mb_sn = pp['m_b_corr'][mask].astype(float)
good = np.isfinite(z_sn) & np.isfinite(mb_sn)
z_sn, mb_sn = z_sn[good], mb_sn[good]
idx = np.argsort(z_sn)
z_sn, mb_sn = z_sn[idx], mb_sn[idx]
with open(COV) as f:
    n_cov = int(f.readline().strip())
    cov_flat = np.loadtxt(f)
cov_full = cov_flat.reshape(n_cov, n_cov)
fi = np.where(mask)[0][good]
fi = fi[np.argsort(pp['zHD'][mask].astype(float)[good])]
C_sn = cov_full[np.ix_(fi, fi)]
L_sn = np.linalg.cholesky(C_sn)
N_SN = len(z_sn)
print(f'Pantheon+: {N_SN} SNe')


In [ ]:
# Cell 4 — BOSS DR12 + DESI DR1
BOSS_Z = np.array([0.38, 0.51, 0.61])
BOSS_DM = np.array([10.27, 13.38, 15.45])
BOSS_DH = np.array([25.00, 22.33, 20.99])
BOSS_FS8 = np.array([0.497, 0.458, 0.436])
BOSS_SDM = np.array([0.15, 0.19, 0.22])
BOSS_SDH = np.array([0.53, 0.38, 0.35])
BOSS_SFS8 = np.array([0.045, 0.038, 0.034])
BOSS_R = np.array([-0.47, -0.43, -0.43])
BOSS_OBS = np.array([BOSS_DM[0],BOSS_DH[0],BOSS_FS8[0],
                      BOSS_DM[1],BOSS_DH[1],BOSS_FS8[1],
                      BOSS_DM[2],BOSS_DH[2],BOSS_FS8[2]])
def build_boss_cov():
    C = np.zeros((9,9))
    for i in range(3):
        j = i*3
        C[j,j] = BOSS_SDM[i]**2
        C[j+1,j+1] = BOSS_SDH[i]**2
        C[j+2,j+2] = BOSS_SFS8[i]**2
        C[j,j+1] = C[j+1,j] = BOSS_R[i]*BOSS_SDM[i]*BOSS_SDH[i]
    return C
BOSS_COV = build_boss_cov()
BOSS_ICOV = np.linalg.inv(BOSS_COV)

DESI_ISO_Z = np.array([0.295, 1.491])
DESI_ISO_DV = np.array([7.93, 26.07])
DESI_ISO_SDV = np.array([0.15, 0.67])
DESI_ANI_Z = np.array([0.510, 0.706, 0.930, 1.317, 2.330])
DESI_ANI_DM = np.array([13.62, 16.85, 21.71, 27.79, 39.71])
DESI_ANI_SDM = np.array([0.25, 0.32, 0.28, 0.69, 0.94])
DESI_ANI_DH = np.array([20.98, 20.08, 17.88, 13.82, 8.52])
DESI_ANI_SDH = np.array([0.61, 0.60, 0.35, 0.42, 0.17])
DESI_ANI_R = np.array([-0.445, -0.420, -0.389, -0.444, -0.477])
print(f'BOSS DR12: 3 bins | DESI DR1: 7 bins')


In [ ]:
# Cell 5 — Configuration (3 free parameters)
import warnings
from scipy.integrate import IntegrationWarning
warnings.filterwarnings('ignore', category=IntegrationWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

N_WALKERS = 24
N_BURN = 500
N_PROD = 3000
BATCH_SIZE = 2
DISP_SECS = 30
CKPT_SECS = 300
CKPT_STEPS_BI = 10
CKPT_STEPS_PROD = 50
GROWTH_NPTS = 600
GROWTH_ZMAX = 50
EEC_NITER =5
CONV_THR = 1e-6

OM_LO, OM_HI = 0.05, 0.35
H0_LO, H0_HI = 40., 100.
S8_LO, S8_HI = 0.4, 1.2

print(f'3 free params: omega_m=[{OM_LO},{OM_HI}] H0=[{H0_LO},{H0_HI}] s8=[{S8_LO},{S8_HI}]')
print(f'Omega_c = Omega_m * {ZHALO_FACTOR:.3f} (derived)')


In [ ]:
# Cell 6 — EEC engine with derived Omega_c
from scipy.integrate import quad, solve_ivp
from scipy.linalg import solve_triangular

def zdrag(omm):
    b1 = 0.313*omm**(-0.419)*(1 + 0.607*omm**0.674)
    b2 = 0.238*omm**0.223
    return 1291.*omm**0.251/(1 + 0.659*omm**0.828)*(1 + b1*omega_b**b2)

def zstar(omm):
    g1 = 0.0783*omega_b**(-0.238)/(1 + 39.5*omega_b**0.763)
    g2 = 0.560/(1 + 21.1*omega_b**1.81)
    return 1048.*(1 + 0.00124*omega_b**(-0.738))*(1 + g1*omm**g2)

def om_or(omm, H0):
    h = H0/100.
    return omm/h**2, omega_r/h**2

def derive_Oc(Om):
    # Omega_c from peak halo formation rate
    return Om * ZHALO_FACTOR

def kappa0(Oc, Om, Or, s8):
    x0 = Om/Oc
    d = x0*np.exp(-x0)*s8**4
    return 1e10 if d < 1e-30 else 2.*(1. - Om - Or)/d

_rs_cache = {}
def rs_phys_Mpc(omm, H0, z_end=None):
    if z_end is None:
        z_end_val = zdrag(omm)
    else:
        z_end_val = z_end
    cache_key = (round(omm, 7), round(H0, 5), round(z_end_val, 2))
    if cache_key in _rs_cache:
        return _rs_cache[cache_key]
    h = H0/100.
    Om = omm/h**2
    Or = omega_r/h**2
    def f(z):
        R = 3.*omega_b/(4.*omega_gam*(1. + z))
        cs = c_km/np.sqrt(3.*(1. + R))
        Hz = H0*np.sqrt(Om*(1+z)**3 + Or*(1+z)**4)
        return cs/Hz
    v, _ = quad(f, z_end_val, 1e6, limit=600, epsabs=1e-8, epsrel=1e-8)
    _rs_cache[cache_key] = v
    return v

def rs_at_zstar(omm, H0):
    return rs_phys_Mpc(omm, H0, zstar(omm))

def solve_growth(Om, Or, Ef, mu=None):
    if mu is None:
        mu = lambda z: 1.
    a0 = 1./(1. + GROWTH_ZMAX)
    a_arr = np.linspace(a0, 1., GROWTH_NPTS)
    def rhs(a, y):
        z = 1./a - 1.
        Ev = Ef(z)
        dz = max(z*1e-5, 1e-6)
        dEdz = ((Ef(z+dz) - Ef(max(z-dz, 0.)))/(2*dz) if z > dz
                else (Ef(dz) - Ef(0.))/dz)
        c1 = 3./a - dEdz/(a**2*Ev)
        c0 = -1.5*Om*mu(z)/(a**5*Ev**2)
        return [y[1], -c1*y[1] - c0*y[0]]
    sol = solve_ivp(rhs, [a0, 1.], [a0, 1.], t_eval=a_arr,
                    method='RK45', rtol=1e-9, atol=1e-11)
    D = sol.y[0]/sol.y[0, -1]
    z_a = (1./a_arr - 1.)[::-1]
    D_a = D[::-1]
    return lambda z: float(np.interp(z, z_a, D_a))

def iterate_EEC(Oc, omm, s8, H0):
    Om, Or = om_or(omm, H0)
    k0 = kappa0(Oc, Om, Or, s8)
    def make_E(Dc):
        def Ef(z):
            x = Om/Oc*(1+z)**3
            Oneq = 0.5*k0*x*np.exp(-x)*Dc(z)**4
            return np.sqrt(max(Om*(1+z)**3 + Or*(1+z)**4 + Oneq, 1e-30))
        return Ef
    E0 = lambda z: np.sqrt(Om*(1+z)**3 + Or*(1+z)**4)
    Dc = lambda z, _D=solve_growth(Om, Or, E0): s8*_D(z)
    for _ in range(EEC_NITER):
        _Dc = Dc
        Ec = make_E(_Dc)
        def mu(z, _D=_Dc):
            rr = Om*(1+z)**3/Oc
            x = Om/Oc*(1+z)**3
            Oneq = 0.5*k0*x*np.exp(-x)*_D(z)**4
            Omz = Om*(1+z)**3
            return 1. if Omz < 1e-30 else 1. + (1. - rr)*Oneq/Omz
        D_new = solve_growth(Om, Or, Ec, mu)
        Dc_new = lambda z, _D=D_new: s8*_D(z)
        diff = max(abs(Dc_new(zz) - _Dc(zz)) for zz in np.linspace(0, 10, 80))
        Dc = Dc_new
        if diff < CONV_THR:
            break
    return Dc, k0, make_E(Dc), Om, Or

def chi_dist(z_up, Ef, H0):
    f = lambda z: c_km/(H0*Ef(z))
    if z_up <= 2:
        v, _ = quad(f, 0, z_up, limit=100)
        return v
    pts = [p for p in [0.5, 2, 10, 100, 500] if p < z_up]
    v, _ = quad(f, 0, z_up, limit=600, points=pts)
    return v

def fsig8(z, Dc, dz=0.01):
    D0 = Dc(z)
    if D0 < 1e-10:
        return 0.
    dDdz = ((Dc(z+dz) - Dc(max(z-dz, 0.)))/(2*dz) if z > dz
            else (Dc(dz) - D0)/dz)
    return -(1. + z)/D0*dDdz*D0

# Test
for H0t in [67., 69., 73.]:
    Om_t = 0.143/(H0t/100.)**2
    Oc_t = derive_Oc(Om_t)
    print(f'  H0={H0t:.0f}  Om={Om_t:.3f}  Oc={Oc_t:.2f}')
print('EEC engine loaded')


In [ ]:
# Cell 7 — Combined likelihood
from scipy.integrate import cumulative_trapezoid

def log_posterior(theta):
    omm, H0, s8 = theta
    if not (OM_LO < omm < OM_HI and H0_LO < H0 < H0_HI and S8_LO < s8 < S8_HI):
        return -np.inf
    Om, Or = om_or(omm, H0)
    if Om + Or >= 1. or Om <= 0 or Or <= 0:
        return -np.inf
    Oc = derive_Oc(Om)
    if Oc <= 0:
        return -np.inf
    try:
        Dc, k0, Ef, Om, Or = iterate_EEC(Oc, omm, s8, H0)
        if abs(Ef(0.) - 1.) > 0.02:
            return -np.inf

        # Distance lookup (cumulative trapezoid, N=2000)
        z_max = max(z_sn[-1]*1.01, 2.5)
        z_fine = np.linspace(0, z_max, 2000)
        E_fine = np.array([Ef(z) for z in z_fine])
        integrand = c_km / (H0 * E_fine)
        chi_cum = np.concatenate([[0], cumulative_trapezoid(integrand, z_fine)])

        rd = rs_phys_Mpc(omm, H0)

        # Planck compressed omega_m (Planck 2018, TT+TE+EE+lowE)
        chi2 = ((omm - omega_m_cmb) / omega_m_cmb_err)**2

        # Pantheon+ SNe (analytic M marginalisation)
        dL_sn = (1 + z_sn) * np.interp(z_sn, z_fine, chi_cum)
        mu_th = 5. * np.log10(np.maximum(dL_sn, 1e-10)) + 25.
        delta = mb_sn - mu_th
        ones = np.ones_like(delta)
        y1 = solve_triangular(L_sn, delta, lower=True)
        y2 = solve_triangular(L_sn, ones, lower=True)
        Cinv_d = solve_triangular(L_sn.T, y1, lower=False)
        Cinv_1 = solve_triangular(L_sn.T, y2, lower=False)
        A = float(delta @ Cinv_d)
        B = float(delta @ Cinv_1)
        D = float(ones @ Cinv_1)
        chi2 += A - B**2 / D

        # BOSS DR12 (BAO + fsig8)
        DM_b = np.array([float(np.interp(z, z_fine, chi_cum)) for z in BOSS_Z]) / rd
        DH_b = c_km / (H0 * np.interp(BOSS_Z, z_fine, E_fine)) / rd
        F8_b = np.array([fsig8(z, Dc) for z in BOSS_Z])
        theory = np.array([DM_b[0], DH_b[0], F8_b[0],
                           DM_b[1], DH_b[1], F8_b[1],
                           DM_b[2], DH_b[2], F8_b[2]])
        dv = BOSS_OBS - theory
        chi2 += float(dv @ BOSS_ICOV @ dv)

        # DESI isotropic
        for i in range(len(DESI_ISO_Z)):
            z = DESI_ISO_Z[i]
            DM_t = float(np.interp(z, z_fine, chi_cum))
            DH_t = c_km / (H0 * float(np.interp(z, z_fine, E_fine)))
            DV_t = (z * DH_t * DM_t**2)**(1./3.) / rd
            chi2 += ((DESI_ISO_DV[i] - DV_t) / DESI_ISO_SDV[i])**2

        # DESI anisotropic
        for i in range(len(DESI_ANI_Z)):
            z = DESI_ANI_Z[i]
            DM_t = float(np.interp(z, z_fine, chi_cum)) / rd
            DH_t = c_km / (H0 * float(np.interp(z, z_fine, E_fine))) / rd
            dDM = DESI_ANI_DM[i] - DM_t
            dDH = DESI_ANI_DH[i] - DH_t
            r = DESI_ANI_R[i]
            s1 = DESI_ANI_SDM[i]
            s2 = DESI_ANI_SDH[i]
            det = (1 - r**2) * s1**2 * s2**2
            chi2 += (dDM**2*s2**2 - 2*r*s1*s2*dDM*dDH + dDH**2*s1**2) / det

        return -0.5 * chi2
    except Exception:
        return -np.inf

# Sanity check
for H0t in [65., 68., 70., 73.]:
    pt = [0.143, H0t, 0.77]
    Om_t = 0.143/(H0t/100.)**2
    Oc_t = derive_Oc(Om_t)
    lp = log_posterior(pt)
    print(f'  H0={H0t:.0f}  Om={Om_t:.3f}  Oc={Oc_t:.2f}  log_p={lp:.2f}')


In [ ]:
# Cell 8 — Checkpoint utilities
import pickle, glob, time, shutil
from IPython.display import clear_output

def save_ckpt(path, obj):
    tmp = path + '.tmp'
    try:
        with open(tmp, 'wb') as f:
            pickle.dump(obj, f, protocol=4)
            f.flush()
            try: os.fsync(f.fileno())
            except: pass
        os.replace(tmp, path)
        return True
    except Exception as e:
        print(f'  WARN: save failed ({e})')
        try:
            if os.path.exists(tmp): os.remove(tmp)
        except: pass
        return False

def load_ckpt(path):
    try:
        with open(path, 'rb') as f:
            return pickle.load(f)
    except Exception:
        try: shutil.move(path, path + '.corrupt')
        except: pass
        return None

def latest_prod():
    files = sorted(glob.glob(os.path.join(CKPT_DIR, 'prod_*.pkl')))
    while files:
        l = files[-1]
        s = int(os.path.basename(l).split('_')[1].split('.')[0])
        try:
            with open(l, 'rb') as f: pickle.load(f)
            return l, s
        except:
            try: shutil.move(l, l + '.corrupt')
            except: pass
            files.pop()
    return None, 0

def latest_burnin():
    files = sorted(glob.glob(os.path.join(CKPT_DIR, 'burnin_step_*.pkl')))
    while files:
        try:
            with open(files[-1], 'rb') as f: pickle.load(f)
            return files[-1]
        except:
            try: shutil.move(files[-1], files[-1] + '.corrupt')
            except: pass
            files.pop()
    return None

def cleanup(pattern, keep=3):
    files = sorted(glob.glob(os.path.join(CKPT_DIR, pattern)))
    for f in files[:-keep]:
        try: os.remove(f)
        except: pass

def dashboard(ch, lp, done, total, t0, phase, af=None):
    n = len(lp)
    if n < 4: return
    recent = ch[max(0, n - max(n//2, 200)):]
    ibest = np.argmax(lp)
    bf = ch[ibest]
    Om_bf, _ = om_or(bf[0], bf[1])
    Oc_bf = derive_Oc(Om_bf)
    S8_bf = bf[2]*np.sqrt(Om_bf/0.3)
    elapsed = (time.time() - t0)/60.
    rate = done/max(elapsed*60, 1)
    eta = (total - done)/max(rate/60, 0.001)
    pct = done/total
    bar = chr(9608)*int(30*pct) + chr(9617)*(30 - int(30*pct))
    W = 72
    def R(s):
        return f'\u2551  {s:<{W-4}}\u2551'
    L = [f'\u2554{"\u2550"*W}\u2557']
    L.append(R('EEC v5 \u2014 Derived (3 free params, \u03a9_c derived)'))
    L.append(R('Pantheon+ \u00b7 BOSS DR12 \u00b7 DESI DR1 \u00b7 Planck \u03c9_m'))
    L.append(f'\u2560{"\u2550"*W}\u2563')
    L.append(R(f'{phase} [{bar}] {pct*100:5.1f}%'))
    L.append(R(f'Step {done:>6}/{total:<6}   {elapsed:.1f}min   ETA {eta:.0f}min' +
               (f'   accept={af:.3f}' if af else '')))
    L.append(f'\u2560{"\u2550"*W}\u2563')
    L.append(R(f'RUNNING ESTIMATES (last {len(recent)} samples)'))
    L.append(R(f'{"Param":>12}  {"Mean":>9}  {"Std":>8}  {"16th":>9}  {"84th":>9}'))
    L.append(R('\u2500'*58))
    for i, lab in enumerate(['omega_m', 'H0', 'sigma_8']):
        v = recent[:, i]
        L.append(R(f'{lab:>12}  {np.mean(v):9.4f}  {np.std(v):8.4f}  {np.percentile(v,16):9.4f}  {np.percentile(v,84):9.4f}'))
    L.append(R('\u2500'*58))
    s = min(100, len(recent))
    Om_r = np.mean([om_or(c[0], c[1])[0] for c in recent[-s:]])
    Oc_r = derive_Oc(Om_r)
    Or_r = np.mean([om_or(c[0], c[1])[1] for c in recent[-s:]])
    S8_r = np.mean([c[2]*np.sqrt(om_or(c[0], c[1])[0]/0.3) for c in recent[-s:]])
    L.append(R(f'{"Omega_c":>12}  {Oc_r:9.4f}  \u2190 derived (z_halo={Z_HALO})'))
    L.append(R(f'{"Omega_m":>12}  {Om_r:9.4f}  \u2190 derived'))
    L.append(R(f'{"Omega_neq":>12}  {1. - Om_r - Or_r:9.4f}  \u2190 derived'))
    L.append(R(f'{"S8":>12}  {S8_r:9.4f}  \u2190 derived'))
    L.append(R(f'{"H0 mean":>12}  {np.mean(recent[-s:,1]):9.3f}'))
    L.append(f'\u2560{"\u2550"*W}\u2563')
    L.append(R(f'BEST log-p={lp[ibest]:.2f}   chi2={-2*lp[ibest]:.1f}'))
    L.append(R(f'  \u03c9_m={bf[0]:.5f}  H\u2080={bf[1]:.3f}  \u03c3\u2088={bf[2]:.4f}'))
    L.append(R(f'  \u03a9_c={Oc_bf:.4f}  \u03a9_m={Om_bf:.4f}  S\u2088={S8_bf:.4f}'))
    L.append(f'\u255a{"\u2550"*W}\u255d')
    clear_output(wait=True)
    print('\n'.join(L))

CKPT_BI = os.path.join(CKPT_DIR, 'burnin.pkl')
CKPT_PFMT = os.path.join(CKPT_DIR, 'prod_{:07d}.pkl')
CKPT_DONE = os.path.join(CKPT_DIR, 'final.pkl')
print('Checkpoint utilities ready')


In [ ]:
# Cell 9 — MCMC (3 free parameters)
import warnings
warnings.filterwarnings('ignore')
ndim = 3

if os.path.exists(CKPT_DONE):
    r = load_ckpt(CKPT_DONE)
    if r and 'chain' in r:
        chain = r['chain']
        lp_chain = r['lp_chain']
        print(f'Loaded: {chain.shape}')
    else:
        chain = None
else:
    chain = None

if chain is None:
    t0 = time.time()
    state = None

    # Burn-in resume
    if os.path.exists(CKPT_BI):
        r = load_ckpt(CKPT_BI)
        if r and 'state' in r:
            state = r['state']
            bi_done = N_BURN
    if state is None:
        lb = latest_burnin()
        if lb:
            r = load_ckpt(lb)
            if r:
                state = r['state']
                bi_done = r['bi_done']
    if state is None:
        bi_done = 0
        p0c = np.array([0.143, 69.0, 0.77])
        p0s = np.array([0.005, 3.0, 0.06])
        p0 = p0c + p0s*np.random.randn(N_WALKERS, ndim)
        p0[:, 0] = np.clip(p0[:, 0], OM_LO + 0.005, OM_HI - 0.005)
        p0[:, 1] = np.clip(p0[:, 1], H0_LO + 1., H0_HI - 1.)
        p0[:, 2] = np.clip(p0[:, 2], S8_LO + 0.02, S8_HI - 0.02)
        valid = []
        for pw in p0:
            if np.isfinite(log_posterior(pw)):
                valid.append(pw)
            else:
                for _ in range(400):
                    pw2 = p0c + p0s*np.random.randn(ndim)*1.5
                    pw2 = np.clip(pw2,
                                  [OM_LO + 0.005, H0_LO + 1., S8_LO + 0.02],
                                  [OM_HI - 0.005, H0_HI - 1., S8_HI - 0.02])
                    if np.isfinite(log_posterior(pw2)):
                        valid.append(pw2)
                        break
        state = np.array(valid[:N_WALKERS])
        print(f'{len(state)} walkers')

    # Burn-in
    if bi_done < N_BURN:
        samp = emcee.EnsembleSampler(N_WALKERS, ndim, log_posterior)
        last_d = last_c = time.time()
        print(f'Burn-in: {N_BURN - bi_done} steps from {bi_done}')
        while bi_done < N_BURN:
            chunk = min(BATCH_SIZE, N_BURN - bi_done)
            state = samp.run_mcmc(state, chunk, progress=False)
            bi_done += chunk
            now = time.time()
            if now - last_d >= DISP_SECS or bi_done >= N_BURN:
                dashboard(samp.get_chain(flat=True),
                          samp.get_log_prob(flat=True),
                          bi_done, N_BURN, t0, 'BURN-IN',
                          np.mean(samp.acceptance_fraction))
                last_d = now
            if now - last_c >= CKPT_SECS or bi_done % CKPT_STEPS_BI == 0 or bi_done >= N_BURN:
                save_ckpt(os.path.join(CKPT_DIR, f'burnin_step_{bi_done:05d}.pkl'),
                          {'state': state, 'bi_done': bi_done,
                           'af': np.mean(samp.acceptance_fraction)})
                cleanup('burnin_step_*.pkl', keep=2)
                last_c = now
        clear_output(wait=True)
        print(f'Burn-in done, accept={np.mean(samp.acceptance_fraction):.3f}')
        del samp
    save_ckpt(CKPT_BI, {'state': state, 'af': 0.0})

    # Production
    ckpt_p, steps_done = latest_prod()
    if steps_done > 0:
        pd = load_ckpt(ckpt_p)
        if pd:
            state = pd['state']
            chain_buf = pd['chain_buf']
            lp_buf = pd['lp_buf']
        else:
            steps_done = 0
            chain_buf = np.empty((0, ndim))
            lp_buf = np.empty(0)
    else:
        chain_buf = np.empty((0, ndim))
        lp_buf = np.empty(0)

    steps_rem = N_PROD - steps_done
    if steps_rem > 0:
        sampler = emcee.EnsembleSampler(N_WALKERS, ndim, log_posterior)
        last_d = last_c = time.time()
        bc = []
        bl = []
        print(f'Production: {steps_rem} steps from {steps_done}')
        while steps_rem > 0:
            chunk = min(BATCH_SIZE, steps_rem)
            state = sampler.run_mcmc(state, chunk, progress=False)
            steps_done += chunk
            steps_rem -= chunk
            bc.append(sampler.get_chain(flat=True)[-chunk*N_WALKERS:])
            bl.append(sampler.get_log_prob(flat=True)[-chunk*N_WALKERS:])
            now = time.time()
            if now - last_d >= DISP_SECS or steps_rem == 0:
                bcc = np.concatenate([chain_buf] + bc)
                bll = np.concatenate([lp_buf] + bl)
                dashboard(bcc, bll, steps_done, N_PROD, t0, 'PRODUCTION',
                          np.mean(sampler.acceptance_fraction))
                last_d = now
            if now - last_c >= CKPT_SECS or steps_done % CKPT_STEPS_PROD == 0 or steps_rem == 0:
                if bc:
                    chain_buf = np.concatenate([chain_buf] + bc)
                    lp_buf = np.concatenate([lp_buf] + bl)
                    bc.clear()
                    bl.clear()
                    sampler.reset()
                save_ckpt(CKPT_PFMT.format(steps_done),
                          {'state': state, 'chain_buf': chain_buf,
                           'lp_buf': lp_buf, 'steps_done': steps_done,
                           'af': np.mean(sampler.acceptance_fraction)})
                cleanup('prod_*.pkl', keep=3)
                last_c = now
        if bc:
            chain_buf = np.concatenate([chain_buf] + bc)
            lp_buf = np.concatenate([lp_buf] + bl)

    chain = chain_buf
    lp_chain = lp_buf
    save_ckpt(CKPT_DONE, {'chain': chain, 'lp_chain': lp_chain})
    clear_output(wait=True)
    print(f'DONE: {(time.time()-t0)/60.:.1f}min, {chain.shape}')


In [ ]:
# Cell 10 — Results
ibest = np.argmax(lp_chain)
bf = chain[ibest]
Om_bf, Or_bf = om_or(bf[0], bf[1])
Oc_bf = derive_Oc(Om_bf)
Om_arr = np.array([om_or(c[0], c[1])[0] for c in chain])
Oc_arr = np.array([derive_Oc(om) for om in Om_arr])
S8_arr = chain[:, 2]*np.sqrt(Om_arr/0.3)
OL_arr = 1. - Om_arr - np.array([om_or(c[0], c[1])[1] for c in chain])

print(f'\n{"="*60}')
print(f'  EEC v5 RESULTS')
print(f'  Omega_c = Omega_m * (1+z_halo)^3,  z_halo = {Z_HALO}')
print(f'  3 free parameters (same count as LCDM)')
print(f'{"="*60}')
print(f'  {"Param":>12}  {"Mean":>9}  {"Std":>8}  {"16th":>9}  {"84th":>9}')
print(f'  {"-"*55}')
for i, lab in enumerate(['omega_m', 'H0', 'sigma_8']):
    v = chain[:, i]
    print(f'  {lab:>12}  {np.mean(v):9.4f}  {np.std(v):8.4f}  {np.percentile(v,16):9.4f}  {np.percentile(v,84):9.4f}')
print(f'  {"-"*55}')
for lab, v in [('Omega_c', Oc_arr), ('Omega_m', Om_arr), ('Omega_neq', OL_arr), ('S8', S8_arr)]:
    print(f'  {lab:>12}  {np.mean(v):9.4f}  {np.std(v):8.4f}  {np.percentile(v,16):9.4f}  {np.percentile(v,84):9.4f}')

Dc_bf, _, Ef_bf, _, _ = iterate_EEC(Oc_bf, bf[0], bf[2], bf[1])
rd = rs_phys_Mpc(bf[0], bf[1])
age_v, _ = quad(lambda z: 1./((1+z)*Ef_bf(z)), 0, 50, limit=300)
age = age_v*3.0857e19/(bf[1]*3.1558e16)

# theta* (r_s at z_STAR, full EEC E(z) for chi)
zs = zstar(bf[0])
rs_s = rs_at_zstar(bf[0], bf[1])
chi_s = chi_dist(zs, Ef_bf, bf[1])
th = rs_s / chi_s
th_t = (th - theta_star)/theta_star_err

print(f'\n  Best fit (log p = {lp_chain[ibest]:.2f}, chi2 = {-2*lp_chain[ibest]:.2f}):')
print(f'    om={bf[0]:.5f}  H0={bf[1]:.2f}  s8={bf[2]:.4f}')
print(f'    Oc={Oc_bf:.4f}  Om={Om_bf:.4f}  rd={rd:.2f} Mpc  Age={age:.2f} Gyr')
print(f'    S8={bf[2]*np.sqrt(Om_bf/0.3):.4f}')
print(f'    theta*={th:.6f} (Planck: {theta_star}, {th_t:.1f} sigma)')
print(f'    rd(z_drag)={rd:.2f} Mpc, rs(z_star)={rs_s:.2f} Mpc')

H0m, H0e = np.mean(chain[:, 1]), np.std(chain[:, 1])
S8m, S8e = np.mean(S8_arr), np.std(S8_arr)
print(f'\n  H0 tension: vs SH0ES = {abs(H0m-73.04)/np.sqrt(H0e**2+1.04**2):.1f} sigma,  vs Planck = {abs(H0m-67.36)/np.sqrt(H0e**2+0.54**2):.1f} sigma')
print(f'  S8 tension: vs WL = {abs(S8m-0.762)/np.sqrt(S8e**2+0.024**2):.1f} sigma,  vs Planck = {abs(S8m-0.832)/np.sqrt(S8e**2+0.013**2):.1f} sigma')

# Compare with LCDM chi2
print(f'\n  For comparison: LCDM best-fit chi2 ~ 1410 (3 params)')
print(f'  EEC chi2 = {-2*lp_chain[ibest]:.1f} (3 params)')
print(f'  Delta chi2 = {-2*lp_chain[ibest] - 1410:.1f}')


In [ ]:
# Cell 11 — Corner plot
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import corner

fig = corner.corner(chain,
    labels=[r'$\omega_m$', r'$H_0$', r'$\sigma_8$'],
    quantiles=[0.16, 0.5, 0.84], show_titles=True,
    title_kwargs={'fontsize': 11})
fig.suptitle(f'EEC v5 derived (z_halo={Z_HALO}, 3 free params)',
             y=1.02, fontsize=12)
fig.savefig('eec_v5_corner.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eec_v5_corner.png')


In [ ]:
# Cell 12 — Save
np.save('eec_v5_chain.npy', chain)
np.save('eec_v5_lp.npy', lp_chain)
with open('eec_v5_results.txt', 'w') as f:
    f.write(f'EEC v5 Results\n{"="*50}\n')
    f.write(f'Omega_c = Omega_m * (1+z_halo)^3, z_halo = {Z_HALO}\n')
    f.write(f'3 free parameters (same as LCDM)\n\n')
    for i, lab in enumerate(['omega_m', 'H0', 'sigma_8']):
        v = chain[:, i]
        f.write(f'{lab}: {np.mean(v):.5f} +/- {np.std(v):.5f} [{np.percentile(v,16):.5f}, {np.percentile(v,84):.5f}]\n')
    f.write(f'\nDerived:\n')
    f.write(f'Omega_c: {np.mean(Oc_arr):.4f} +/- {np.std(Oc_arr):.4f}\n')
    f.write(f'Omega_m: {np.mean(Om_arr):.4f} +/- {np.std(Om_arr):.4f}\n')
    f.write(f'S8: {np.mean(S8_arr):.4f} +/- {np.std(S8_arr):.4f}\n')
    f.write(f'\nBest fit: chi2={-2*lp_chain[ibest]:.2f}\n')
    f.write(f'H0={bf[1]:.3f}  S8={bf[2]*np.sqrt(Om_bf/0.3):.4f}  Age={age:.2f} Gyr\n')
print('Saved all outputs')
try:
    from google.colab import files
    for fn in ['eec_v5_results.txt', 'eec_v5_corner.png']:
        try: files.download(fn)
        except: pass
except: pass
